<a href="https://colab.research.google.com/github/harishkulkarni10/ai-research-assistant-rag/blob/dev/notebooks/generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -q sentence-transformers transformers accelerate bitsandbytes

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
!git clone -b dev https://github.com/harishkulkarni10/ai-research-assistant-rag.git

In [ ]:
PROJECT_ROOT = Path("/content/ai-research-assistant-rag")

chunks_path = PROJECT_ROOT / "data" / "processed" / "chunks_bge.parquet"
chunks_df = pd.read_parquet(chunks_path)

print(f"Loaded {len(chunks_df):,} chunks")
print(chunks_df.columns)

In [ ]:
embeddings = torch.tensor(
    np.array(chunks_df["embedding"].tolist(), dtype=np.float32),
    device=device,
)

print("Embeddings tensor shape:", embeddings.shape)

In [ ]:
embedder = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=device
  )

print('BGE query encoder is ready')

In [ ]:
def retrieve_chunks(query: str, top_k: int = 4):
    # Encode query
    q_emb = embedder.encode(query, normalize_embeddings=True)
    q_emb = torch.tensor(q_emb, dtype=torch.float32, device=device).unsqueeze(0)

    # Cosine similarity
    scores = torch.matmul(q_emb, embeddings.T)[0]

    # Top-k on GPU
    top_scores, top_indices = torch.topk(scores, k=top_k)

    top_indices = top_indices.cpu().numpy()
    top_scores = top_scores.cpu().numpy()

    results = chunks_df.iloc[top_indices].copy()
    results["score"] = top_scores

    return results.sort_values("score", ascending=False)


In [ ]:
from transformers import BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config
)

model.eval()
print("Mistral-7B-Instruct loaded successfully (4-bit)")


In [ ]:
def build_prompt(context: str, question: str) -> str:
    return f"""
You are a scientific research assistant.

Answer the question using ONLY the context below.
If the answer is not contained in the context, say:
"Not found in the provided documents."

Context:
{context}

Question:
{question}

Answer:
""".strip()

In [ ]:
def generate_answer(prompt: str, max_new_tokens: int = 300):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("Answer:")[-1].strip()


In [ ]:
queries = [
    "How are nonlinear dynamical systems modeled using differential equations?",
    "What methods are used to analyze stability and phase transitions in physical systems?",
    "How are time series and temporal dynamics analyzed in complex systems?"
]

for q in queries:
    print("=" * 80)
    print("QUESTION:", q)

    retrieved = retrieve_chunks(q, top_k=4)

    context = "\n\n".join(retrieved["text"].tolist())

    prompt = build_prompt(context, q)
    answer = generate_answer(prompt)

    print("\nANSWER:\n", answer)

In [ ]:
import json

results = []
for q in queries:
    retrieved = retrieve_chunks(q, top_k=4)
    context = "\n\n".join(retrieved["text"].tolist())
    prompt = build_prompt(context, q)
    answer = generate_answer(prompt)
    results.append({
        "query": q,
        "retrieved_chunks": retrieved[["chunk_id", "paper_id", "score", "text"]].to_dict(orient="records"),
        "generated_answer": answer
    })

with open("rag_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to rag_results.json")

In [ ]:
!cp /content/05_generation.ipynb /content/ai-research-assistant-rag/notebooks/05_generation/generation.ipynb
!cd /content/ai-research-assistant-rag && git add notebooks/05_generation/generation.ipynb
!cd /content/ai-research-assistant-rag && git commit -m "feat: add generation notebook with Mistral-7B RAG"
!cd /content/ai-research-assistant-rag && git push origin dev